<a href="https://colab.research.google.com/github/nehansa2003/NLP_project/blob/main/Undelying_labeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import os
import re
from google.colab import drive
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
project_folder = '/content/drive/MyDrive/NLP_Game_Review_Project'

print("Project folder:")
print(project_folder)

Project folder:
/content/drive/MyDrive/NLP_Game_Review_Project


In [ ]:
train_df = pd.read_csv(
    os.path.join(project_folder, 'train_reviews.csv')
)

test_df = pd.read_csv(
    os.path.join(project_folder, 'test_reviews.csv')
)

unseen_df = pd.read_csv(
    os.path.join(project_folder, 'unseen_reviews.csv')
)

In [ ]:
print("=" * 50)
print("DATASET SIZES")
print("=" * 50)

print(f"Training : {train_df.shape}")
print(f"Testing  : {test_df.shape}")
print(f"Unseen   : {unseen_df.shape}")

DATASET SIZES
Training : (77607, 14)
Testing  : (19640, 14)
Unseen   : (2780, 14)


In [ ]:
train_labeled = train_df.dropna(subset=['sentiment']).copy()
test_labeled = test_df.dropna(subset=['sentiment']).copy()
unseen_labeled = unseen_df.dropna(subset=['sentiment']).copy()

print("Labeled reviews:")
print("----------------")

print("Training:", len(train_labeled))
print("Testing :", len(test_labeled))
print("Unseen  :", len(unseen_labeled))

Labeled reviews:
----------------
Training: 60603
Testing : 15370
Unseen  : 2171


sentiment distribution

In [ ]:
print("TRAINING SENTIMENT")
print(train_labeled['sentiment'].value_counts())

print("\nTESTING SENTIMENT")
print(test_labeled['sentiment'].value_counts())

print("\nUNSEEN SENTIMENT")
print(unseen_labeled['sentiment'].value_counts())

TRAINING SENTIMENT
sentiment
positive    54190
neutral      6175
negative      238
Name: count, dtype: int64

TESTING SENTIMENT
sentiment
positive    13723
neutral      1596
negative       51
Name: count, dtype: int64

UNSEEN SENTIMENT
sentiment
positive    1952
neutral      209
negative      10
Name: count, dtype: int64


Text Lables

In [ ]:
X_train = train_labeled['clean_review_text']
y_train = train_labeled['sentiment']

X_test = test_labeled['clean_review_text']
y_test = test_labeled['sentiment']

X_unseen = unseen_labeled['clean_review_text']
y_unseen = unseen_labeled['sentiment']

print("Training:", len(X_train))
print("Testing :", len(X_test))
print("Unseen  :", len(X_unseen))

Training: 60603
Testing : 15370
Unseen  : 2171


In [ ]:
print("Empty training texts:", X_train.isna().sum())
print("Empty testing texts :", X_test.isna().sum())
print("Empty unseen texts  :", X_unseen.isna().sum())

print("Blank training texts:",
      (X_train.str.strip() == '').sum())

print("Blank testing texts:",
      (X_test.str.strip() == '').sum())

print("Blank unseen texts:",
      (X_unseen.str.strip() == '').sum())

Empty training texts: 0
Empty testing texts : 0
Empty unseen texts  : 0
Blank training texts: 0
Blank testing texts: 0
Blank unseen texts: 0


TF-IDF Setting

In [ ]:
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    max_features=150000,
    sublinear_tf=True
)

Training Data

In [ ]:
X_train_tfidf = tfidf.fit_transform(X_train)

print("Training TF-IDF shape:", X_train_tfidf.shape)

Training TF-IDF shape: (60603, 93663)


Transform testing and unseen data

In [ ]:
X_test_tfidf = tfidf.transform(X_test)
X_unseen_tfidf = tfidf.transform(X_unseen)

print("Training:", X_train_tfidf.shape)
print("Testing :", X_test_tfidf.shape)
print("Unseen  :", X_unseen_tfidf.shape)

Training: (60603, 93663)
Testing : (15370, 93663)
Unseen  : (2171, 93663)


Memory Checking

In [ ]:
print("TF-IDF matrix type:", type(X_train_tfidf))
print("Number of features:", X_train_tfidf.shape[1])

TF-IDF matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Number of features: 93663


underlying labeling

In [ ]:
inspection_df = pd.concat(
    [train_df, test_df, unseen_df],
    ignore_index=True
)

print("Inspection dataset:", inspection_df.shape)

Inspection dataset: (100027, 14)


In [ ]:
print("ALL RAW SCORE VALUES")
print("=" * 60)

score_counts = (
    inspection_df['score']
    .value_counts(dropna=False)
    .reset_index()
)

score_counts.columns = ['score', 'count']

display(score_counts.head(100))

ALL RAW SCORE VALUES


,score,count
0,9 / 10.0,13281
1,8 / 10.0,9046
2,5 stars,8733
3,8.5 / 10.0,7963
4,10-Aug,5270
...,...,...
95,Liked,63
96,65 / 100,63
97,87%,63
98,A or higher,61


Inspect normalized scores

In [ ]:
print("NORMALIZED SCORE DISTRIBUTION")
print("=" * 60)

display(
    inspection_df['score_normalized'].describe()
)

NORMALIZED SCORE DISTRIBUTION


,score_normalized
count,78144.000000
mean,87.786932
std,12.931056
min,7.000000
25%,80.000000
50%,88.000000
75%,93.000000
max,200.000000


Inspect the lowest scores

In [ ]:
print("LOWEST SCORES")
print("=" * 60)

display(
    inspection_df[
        [
            'game_name',
            'score',
            'score_normalized',
            'sentiment',
            'review_text'
        ]
    ]
    .sort_values(
        'score_normalized',
        ascending=True
    )
    .head(30)
)

LOWEST SCORES


,game_name,score,score_normalized,sentiment,review_text
41368,Gal Guardians: Servants of the Dark,7 / 100,7.0,negative,Gal Guardians: Servants of the Dark is a good ...
99760,NeverAwake,9 / 100,9.0,negative,NeverAwake kept me awake long hours as I felt ...
55567,Watch Dogs 2,2 / 10.0,20.0,negative,"Watch_Dogs 2 heads in a lot of directions, and..."
51921,Diablo IV: Lord of Hatred,2.5 / 10.0,25.0,negative,Even if the core experience is still that of a...
72095,Blood & Truth,2.9 / 10.0,29.0,negative,Blood & Truth's story is about as gripping as ...
54762,Torn Away,3 / 10.0,30.0,negative,"War is a difficult subject to cover, and unfor..."
54237,World of Goo 2,3 / 10.0,30.0,negative,"The slow pacing, along with unskippable story ..."
50005,Death Stranding,3 / 10.0,30.0,negative,"Death Stranding is not entertaining. As such, ..."
66403,MARVEL Cosmic Invasion,3 / 10.0,30.0,negative,"At $29.99, the price is frigging obscene for j..."
82361,Dragon's Dogma 2,3 / 10.0,30.0,negative,Dragon’s Dogma 2 is outwardly hostile to its a...


Inspect reviews without sentiment

In [ ]:
print("REVIEWS CURRENTLY WITHOUT SENTIMENT")
print("=" * 60)

unlabeled_inspection = inspection_df[
    inspection_df['sentiment'].isna()
]

print("Unlabeled reviews:", len(unlabeled_inspection))

display(
    unlabeled_inspection[
        [
            'game_name',
            'score',
            'score_normalized',
            'review_text'
        ]
    ].head(50)
)

REVIEWS CURRENTLY WITHOUT SENTIMENT
Unlabeled reviews: 21883


,game_name,score,score_normalized,review_text
6,Super Mario Odyssey,10-Oct,NaN,Super Mario Odyssey is a massive and magnifice...
8,Super Mario Odyssey,NaN,NaN,Every element gelled so well that I was simply...
9,Super Mario Odyssey,NaN,NaN,"Odyssey is the best Mario game in many, many y..."
10,Super Mario Odyssey,NaN,NaN,"If you have a Switch, get this game. If you do..."
12,Super Mario Odyssey,10-Oct,NaN,Super Mario Odyssey represents a shift in dire...
15,Super Mario Odyssey,10-Oct,NaN,"It's extraordinary, really, that after all thi..."
20,Super Mario Odyssey,10-Sep,NaN,Super Mario Odyssey is an incredible game for ...
21,Super Mario Odyssey,NaN,NaN,This is one of the closest things to the fount...
25,Super Mario Odyssey,10-Oct,NaN,A joyful celebration – and evolution – of the ...
26,Super Mario Odyssey,10-Oct,NaN,Super Mario Odyssey is bursting at the seams w...


Find the unusual score formats

In [ ]:
print("RAW SCORE VALUES THAT BECAME NaN")
print("=" * 60)

nan_scores = inspection_df[
    inspection_df['score_normalized'].isna() &
    inspection_df['score'].notna()
]

display(
    nan_scores['score']
    .value_counts()
    .head(100)
)

RAW SCORE VALUES THAT BECAME NaN


,count
score,
10-Aug,5270
10-Sep,4950
10-Jul,1745
10-Oct,919
Recommended,831
...,...
59%,1
40%,1
54%,1


identify the scores producing >100

In [ ]:
print("SCORES ABOVE 100")
print("=" * 60)

display(
    inspection_df[
        inspection_df['score_normalized'] > 100
    ][
        ['score', 'score_normalized', 'game_name', 'review_text']
    ].head(50)
)

print("\nCounts:")
print(
    inspection_df[
        inspection_df['score_normalized'] > 100
    ]['score'].value_counts().head(30)
)

SCORES ABOVE 100


,score,score_normalized,game_name,review_text
86,10 stars,200.0,Super Mario Odyssey,It's been a long time since we've seen a good ...
403,10 stars,200.0,The Legend of Zelda: Breath of the Wild,This latest Zelda installment is a real breath...
597,10 stars,200.0,Red Dead Redemption 2,Red Dead Redemption 2 on PC is truly mesmerizi...
598,10 stars,200.0,Red Dead Redemption 2,The purposefully slow pacing and clunky contro...
645,10 stars,200.0,Red Dead Redemption 2,Red Dead Redemption 2 feels like Rockstar has ...
761,10 stars,200.0,The Legend of Zelda: Tears of the Kingdom,"In my opinion, the answer is yes. I’ve dumped ..."
954,10 stars,200.0,Elden Ring,Almost nothing compares to Elden Ring in quali...
1308,10 stars,200.0,The Last of Us Remastered,For me The Last of Us: Remastered is the best ...
1455,10 stars,200.0,God of War,Deep with allegory and underpinned by rich com...
1921,10 stars,200.0,Metroid Prime Remastered,"With enhanced graphics and modern controls, Me..."



Counts:
score
10 stars    545
9 stars       3
Name: count, dtype: int64


score-normalization function

In [ ]:

def normalize_score_v2(score):
    if pd.isna(score):
        return np.nan

    score = str(score).strip().lower()

    # ------------------------------------------------
    # 1. Date-converted scores
    # Examples:
    # 10-jul -> 7/10
    # 10-aug -> 8/10
    # 10-sep -> 9/10
    # 10-oct -> 10/10
    # ------------------------------------------------

    month_score_map = {
        '10-jul': 70,
        '10-aug': 80,
        '10-sep': 90,
        '10-oct': 100
    }

    if score in month_score_map:
        return month_score_map[score]

    # ------------------------------------------------
    # 2. Fractions such as:
    # 9 / 10
    # 9.8 / 10.0
    # 65 / 100
    # 4.2 / 5.0
    # ------------------------------------------------

    match = re.search(
        r'(\d+(?:\.\d+)?)\s*/\s*(\d+(?:\.\d+)?)',
        score
    )

    if match:
        value = float(match.group(1))
        maximum = float(match.group(2))

        if maximum > 0:
            normalized = (value / maximum) * 100

            if 0 <= normalized <= 100:
                return normalized

    # ------------------------------------------------
    # 3. Percentages
    # Examples:
    # 87%
    # 59%
    # 40%
    # ------------------------------------------------

    match = re.search(
        r'(\d+(?:\.\d+)?)\s*%',
        score
    )

    if match:
        value = float(match.group(1))

        if 0 <= value <= 100:
            return value

    # ------------------------------------------------
    # 4. Star ratings
    # Examples:
    # 5 stars
    # 4 stars
    # 3.5 stars
    # ------------------------------------------------

    match = re.search(
        r'(\d+(?:\.\d+)?)\s*stars?',
        score
    )

    if match:
        value = float(match.group(1))

        if 0 <= value <= 5:
            return (value / 5) * 100

    # ------------------------------------------------
    # 5. Plain numeric scores
    # ------------------------------------------------

    if re.fullmatch(r'\d+(?:\.\d+)?', score):

        value = float(score)

        # Scores 0-10
        if 0 <= value <= 10:
            return value * 10

        # Scores 0-100
        if 10 < value <= 100:
            return value

    # ------------------------------------------------
    # 6. Qualitative verdicts
    # ------------------------------------------------

    qualitative_scores = {
        'essential': 100,
    }

    if score in qualitative_scores:
        return qualitative_scores[score]

    # Unknown qualitative scores remain NaN
    return np.nan

In [ ]:
inspection_df['score_normalized_v2'] = (
    inspection_df['score'].apply(normalize_score_v2)
)

In [ ]:
def create_sentiment_v2(score):
    if pd.isna(score):
        return np.nan

    if score >= 80:
        return 'positive'
    elif score >= 60:
        return 'neutral'
    else:
        return 'negative'


inspection_df['sentiment_v2'] = (
    inspection_df['score_normalized_v2']
    .apply(create_sentiment_v2)
)

In [ ]:
print("OLD LABEL COUNTS")
print("=" * 50)
print(
    inspection_df['sentiment']
    .value_counts(dropna=False)
)

print("\nNEW LABEL COUNTS")
print("=" * 50)
print(
    inspection_df['sentiment_v2']
    .value_counts(dropna=False)
)

OLD LABEL COUNTS
sentiment
positive    69865
NaN         21883
neutral      7980
negative      299
Name: count, dtype: int64

NEW LABEL COUNTS
sentiment_v2
positive    82288
neutral      9959
NaN          7468
negative      312
Name: count, dtype: int64


In [ ]:
old_labeled = inspection_df['sentiment'].notna().sum()
new_labeled = inspection_df['sentiment_v2'].notna().sum()

print("Previously labeled :", old_labeled)
print("Now labeled        :", new_labeled)
print("New labels recovered:", new_labeled - old_labeled)

Previously labeled : 78144
Now labeled        : 92559
New labels recovered: 14415


Check the new score distribution

In [ ]:
print(
    inspection_df['score_normalized_v2'].describe()
)

count    92559.000000
mean        86.539062
std          8.814409
min          7.000000
25%         80.000000
50%         87.000000
75%         90.000000
max        100.000000
Name: score_normalized_v2, dtype: float64


Inspect the remaining unlabeled scores

In [ ]:
remaining_unlabeled = inspection_df[
    inspection_df['score_normalized_v2'].isna() &
    inspection_df['score'].notna()
]

print(
    "Remaining non-empty scores without numeric normalization:",
    len(remaining_unlabeled)
)

display(
    remaining_unlabeled['score']
    .value_counts()
    .head(100)
)

Remaining non-empty scores without numeric normalization: 4251


,count
score,
Recommended,831
10 stars,545
10-Jun,390
Yes,277
Worth your time,256
Loved,232
Liked-a-lot,202
5-Apr,201
Buy,173


Improve the parser again

In [ ]:
def normalize_score_v3(score):
    if pd.isna(score):
        return np.nan

    score = str(score).strip().lower()

    # ---------------------------------------------
    # 1. Excel/date-converted scores
    # ---------------------------------------------

    date_score_map = {
        '10-jul': 70,
        '10-aug': 80,
        '10-sep': 90,
        '10-oct': 100,

        '10-jun': 60,
        '10-may': 50,
        '10-apr': 40,
        '10-mar': 30,
    }

    if score in date_score_map:
        return date_score_map[score]

    # ---------------------------------------------
    # 2. Fractions: 9/10, 4.2/5, 65/100
    # ---------------------------------------------

    match = re.search(
        r'(\d+(?:\.\d+)?)\s*/\s*(\d+(?:\.\d+)?)',
        score
    )

    if match:
        value = float(match.group(1))
        maximum = float(match.group(2))

        if maximum > 0:
            normalized = (value / maximum) * 100

            if 0 <= normalized <= 100:
                return normalized

    # ---------------------------------------------
    # 3. Percentages
    # ---------------------------------------------

    match = re.search(
        r'(\d+(?:\.\d+)?)\s*%',
        score
    )

    if match:
        value = float(match.group(1))

        if 0 <= value <= 100:
            return value

    # ---------------------------------------------
    # 4. Star ratings
    #
    # "5 stars"
    # "10 stars"
    #
    # 5-star scale -> 0-100
    # 10-star scale -> 0-100
    # ---------------------------------------------

    match = re.search(
        r'(\d+(?:\.\d+)?)\s*stars?',
        score
    )

    if match:
        value = float(match.group(1))

        if 0 <= value <= 5:
            return (value / 5) * 100

        if 5 < value <= 10:
            return (value / 10) * 100

    # ---------------------------------------------
    # 5. Plain numeric scores
    # ---------------------------------------------

    if re.fullmatch(r'\d+(?:\.\d+)?', score):

        value = float(score)

        if 0 <= value <= 10:
            return value * 10

        if 10 < value <= 100:
            return value

    # ---------------------------------------------
    # 6. Letter grades
    #
    # Standard approximate academic-style mapping
    # ---------------------------------------------

    grade_scores = {
        'a+': 100,
        'a': 95,
        'a-': 90,
        'b+': 85,
        'b': 80,
        'b-': 75,
        'c+': 70,
        'c': 65,
        'c-': 60,
        'd+': 55,
        'd': 50,
        'd-': 45,
        'f': 30
    }

    if score in grade_scores:
        return grade_scores[score]

    # ---------------------------------------------
    # 7. Clear positive verdicts
    # ---------------------------------------------

    positive_verdicts = {
        'essential': 100,
        'masterpiece': 100,
        'mind-blown': 95,
        'loved': 95,
        'liked-a-lot': 90,
        'liked': 85,
        'recommended': 85,
        'buy': 85,
        'worth your time': 85,
        'a or higher': 95,
        'yes': 85
    }

    if score in positive_verdicts:
        return positive_verdicts[score]

    # ---------------------------------------------
    # 8. Clear negative verdicts
    # ---------------------------------------------

    negative_verdicts = {
        'avoid': 20,
        "don't buy": 20,
        'no': 20,
        'disliked': 30,
    }

    if score in negative_verdicts:
        return negative_verdicts[score]

    # ---------------------------------------------
    # 9. Ambiguous verdicts remain unlabeled
    # ---------------------------------------------

    return np.nan

Generate version 3 labels

In [ ]:
inspection_df['score_normalized_v3'] = (
    inspection_df['score'].apply(normalize_score_v3)
)

inspection_df['sentiment_v3'] = (
    inspection_df['score_normalized_v3']
    .apply(create_sentiment_v2)
)

Compare all three versions

In [ ]:
print("LABEL COMPARISON")
print("=" * 60)

comparison_labels = pd.DataFrame({
    'Original': inspection_df['sentiment'].value_counts(),
    'Version 2': inspection_df['sentiment_v2'].value_counts(),
    'Version 3': inspection_df['sentiment_v3'].value_counts()
})

display(comparison_labels)

LABEL COMPARISON


,Original,Version 2,Version 3
positive,69865,82288,85426
neutral,7980,9959,10360
negative,299,312,439


In [ ]:
print("LABELED REVIEW COUNTS")
print("=" * 60)

print(
    "Original:",
    inspection_df['sentiment'].notna().sum()
)

print(
    "Version 2:",
    inspection_df['sentiment_v2'].notna().sum()
)

print(
    "Version 3:",
    inspection_df['sentiment_v3'].notna().sum()
)

LABELED REVIEW COUNTS
Original: 78144
Version 2: 92559
Version 3: 96225


Check the new class distribution

In [ ]:
print("VERSION 3 SENTIMENT DISTRIBUTION")
print("=" * 60)

sentiment_v3_counts = (
    inspection_df['sentiment_v3']
    .value_counts(dropna=False)
)

display(sentiment_v3_counts)

print("\nPercentages:")
display(
    inspection_df['sentiment_v3']
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

VERSION 3 SENTIMENT DISTRIBUTION


,count
sentiment_v3,
positive,85426
neutral,10360
NaN,3802
negative,439



Percentages:


,proportion
sentiment_v3,
positive,85.40
neutral,10.36
NaN,3.80
negative,0.44


Update the sentiment column

In [ ]:
# Create the final sentiment column
inspection_df['sentiment_final'] = inspection_df['sentiment_v3']

print("Final sentiment distribution")
print("=" * 60)

display(
    inspection_df['sentiment_final']
    .value_counts(dropna=False)
)

Final sentiment distribution


,count
sentiment_final,
positive,85426
neutral,10360
NaN,3802
negative,439


Separate labeled and unlabeled reviews

In [ ]:
labeled_df = inspection_df[
    inspection_df['sentiment_final'].notna()
].copy()

unlabeled_df = inspection_df[
    inspection_df['sentiment_final'].isna()
].copy()

print("Labeled reviews  :", len(labeled_df))
print("Unlabeled reviews:", len(unlabeled_df))

Labeled reviews  : 96225
Unlabeled reviews: 3802


Preserve the original game split

In [ ]:
train_game_ids = set(train_df['game_id'].unique())
test_game_ids = set(test_df['game_id'].unique())
unseen_game_ids = set(unseen_df['game_id'].unique())

print("Training games :", len(train_game_ids))
print("Testing games  :", len(test_game_ids))
print("Unseen games   :", len(unseen_game_ids))

Training games : 1550
Testing games  : 400
Unseen games   : 50


Create the corrected datasets

In [ ]:
train_final = labeled_df[
    labeled_df['game_id'].isin(train_game_ids)
].copy()

test_final = labeled_df[
    labeled_df['game_id'].isin(test_game_ids)
].copy()

unseen_final = labeled_df[
    labeled_df['game_id'].isin(unseen_game_ids)
].copy()

Verify the game split

In [ ]:
print("FINAL DATASET SIZES")
print("=" * 60)

print("Training reviews :", len(train_final))
print("Testing reviews  :", len(test_final))
print("Unseen reviews   :", len(unseen_final))

print("\nUnique games:")
print("Training games :", train_final['game_id'].nunique())
print("Testing games  :", test_final['game_id'].nunique())
print("Unseen games   :", unseen_final['game_id'].nunique())

FINAL DATASET SIZES
Training reviews : 74665
Testing reviews  : 18900
Unseen reviews   : 2660

Unique games:
Training games : 1550
Testing games  : 400
Unseen games   : 50


Check sentiment distribution in each split

In [ ]:
print("TRAINING SENTIMENT")
print("=" * 60)
display(train_final['sentiment_final'].value_counts())

print("\nTESTING SENTIMENT")
print("=" * 60)
display(test_final['sentiment_final'].value_counts())

print("\nUNSEEN SENTIMENT")
print("=" * 60)
display(unseen_final['sentiment_final'].value_counts())

TRAINING SENTIMENT


,count
sentiment_final,
positive,66274
neutral,8037
negative,354



TESTING SENTIMENT


,count
sentiment_final,
positive,16775
neutral,2053
negative,72



UNSEEN SENTIMENT


,count
sentiment_final,
positive,2377
neutral,270
negative,13


Check for game leakage

In [ ]:
train_games = set(train_final['game_id'])
test_games = set(test_final['game_id'])
unseen_games = set(unseen_final['game_id'])

print("Train ∩ Test   :", len(train_games & test_games))
print("Train ∩ Unseen :", len(train_games & unseen_games))
print("Test ∩ Unseen  :", len(test_games & unseen_games))

Train ∩ Test   : 0
Train ∩ Unseen : 0
Test ∩ Unseen  : 0


Save the corrected datasets

In [ ]:
train_final.to_csv(
    '/content/train_reviews_v2.csv',
    index=False
)

test_final.to_csv(
    '/content/test_reviews_v2.csv',
    index=False
)

unseen_final.to_csv(
    '/content/unseen_reviews_v2.csv',
    index=False
)

print("✓ Corrected datasets saved successfully.")

✓ Corrected datasets saved successfully.


In [ ]:
project_path = '/content/drive/MyDrive/Game_Review_NLP'

os.makedirs(project_path, exist_ok=True)

print("Project folder:", project_path)

Project folder: /content/drive/MyDrive/Game_Review_NLP


In [ ]:
import shutil

shutil.copy(
    '/content/train_reviews_v2.csv',
    project_path
)

shutil.copy(
    '/content/test_reviews_v2.csv',
    project_path
)

shutil.copy(
    '/content/unseen_reviews_v2.csv',
    project_path
)

print("✓ Files copied to Google Drive.")

✓ Files copied to Google Drive.


In [ ]:
inspection_df.to_csv(
    f'{project_path}/all_reviews_processed_v3.csv',
    index=False
)

print("✓ Complete processed dataset saved.")

✓ Complete processed dataset saved.
